In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predecir-la-desercion-escolar-y-el-exito-academico/train (2).csv
/kaggle/input/predecir-la-desercion-escolar-y-el-exito-academico/test (1).csv


In [ ]:
#Miguel Cornejo, Nicolas Arellano
#V25
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, 
                               GradientBoostingClassifier, AdaBoostClassifier,
                               BaggingClassifier, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# =============================================================================
# CARGA
# =============================================================================

train_df = pd.read_csv('/kaggle/input/predecir-la-desercion-escolar-y-el-exito-academico/train (2).csv')
test_df = pd.read_csv('/kaggle/input/predecir-la-desercion-escolar-y-el-exito-academico/test (1).csv')

test_ids = test_df['id'].values if 'id' in test_df.columns else range(len(test_df))
X_train = train_df.drop('Target', axis=1).values
y_train = train_df['Target'].values
X_test = test_df.drop('id', axis=1).values if 'id' in test_df.columns else test_df.values

le = LabelEncoder()
y_encoded = le.fit_transform(y_train)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f" TRAIN: {len(X_train)} | TEST: {len(X_test)}")

# =============================================================================
# MODELOS AMPLIADOS (MÁS QUE V18)
# =============================================================================

print("\n PREPARANDO MODELOS AMPLIADOS")

def create_all_models():
    """Crear configuraciones ampliadas"""
    models = []
    
    # ===== XGBoost (40 variantes - +10 vs V18) =====
    for n_est in [800, 1000, 1200, 1500, 1800]:
        for max_d in [5, 6, 7, 8]:
            for lr in [0.015, 0.02, 0.025, 0.03]:
                for reg in [0.2, 0.3, 0.5]:
                    if len([m for m in models if m[0] == 'XGB']) < 40:
                        models.append(('XGB', {
                            'n_estimators': n_est, 'max_depth': max_d, 'learning_rate': lr,
                            'subsample': 0.8, 'colsample_bytree': 0.8,
                            'reg_alpha': reg, 'reg_lambda': reg, 'min_child_weight': 2,
                            'random_state': 42, 'eval_metric': 'mlogloss', 'use_label_encoder': False
                        }, xgb.XGBClassifier))
    
    # ===== LightGBM (40 variantes - +10 vs V18) =====
    for n_est in [800, 1000, 1200, 1500, 1800]:
        for leaves in [31, 40, 50, 60]:
            for lr in [0.015, 0.02, 0.025, 0.03]:
                for cw in ['balanced', None]:
                    if len([m for m in models if m[0] == 'LGB']) < 40:
                        models.append(('LGB', {
                            'n_estimators': n_est, 'num_leaves': leaves, 'max_depth': 6,
                            'learning_rate': lr, 'subsample': 0.8, 'colsample_bytree': 0.8,
                            'reg_alpha': 0.5, 'reg_lambda': 0.5, 'min_child_samples': 15,
                            'class_weight': cw, 'random_state': 42, 'verbose': -1
                        }, lgb.LGBMClassifier))
    
    # ===== CatBoost (30 variantes - +10 vs V18) =====
    for n_est in [800, 1000, 1200, 1500, 1800]:
        for depth in [5, 6, 7, 8]:
            for lr in [0.015, 0.02, 0.025, 0.03]:
                if len([m for m in models if m[0] == 'CAT']) < 30:
                    models.append(('CAT', {
                        'iterations': n_est, 'depth': depth, 'learning_rate': lr,
                        'l2_leaf_reg': 3, 'random_seed': 42, 'verbose': 0
                    }, CatBoostClassifier))
    
    # ===== Random Forest (20 variantes - +5 vs V18) =====
    for n_est in [300, 500, 700, 900]:
        for max_d in [10, 15, 20, 25]:
            for cw in ['balanced', None]:
                if len([m for m in models if m[0] == 'RF']) < 20:
                    models.append(('RF', {
                        'n_estimators': n_est, 'max_depth': max_d,
                        'min_samples_split': 5, 'min_samples_leaf': 2,
                        'max_features': 0.7, 'class_weight': cw,
                        'random_state': 42, 'n_jobs': -1
                    }, RandomForestClassifier))
    
    # ===== Extra Trees (15 variantes - +5 vs V18) =====
    for n_est in [300, 500, 700]:
        for max_d in [10, 15, 20, 25]:
            if len([m for m in models if m[0] == 'ET']) < 15:
                models.append(('ET', {
                    'n_estimators': n_est, 'max_depth': max_d,
                    'min_samples_split': 5, 'class_weight': 'balanced',
                    'random_state': 42, 'n_jobs': -1
                }, ExtraTreesClassifier))
    
    # ===== Gradient Boosting (15 variantes - +5 vs V18) =====
    for n_est in [300, 500, 700]:
        for lr in [0.05, 0.1, 0.15]:
            for max_d in [4, 5, 6]:
                if len([m for m in models if m[0] == 'GB']) < 15:
                    models.append(('GB', {
                        'n_estimators': n_est, 'learning_rate': lr, 'max_depth': max_d,
                        'subsample': 0.8, 'random_state': 42
                    }, GradientBoostingClassifier))
    
    # ===== Hist Gradient Boosting (15 variantes - +5 vs V18) =====
    for lr in [0.05, 0.1, 0.15]:
        for max_d in [5, 10, 15, 20, 25]:
            if len([m for m in models if m[0] == 'HGB']) < 15:
                models.append(('HGB', {
                    'learning_rate': lr, 'max_depth': max_d,
                    'max_iter': 300, 'random_state': 42
                }, HistGradientBoostingClassifier))
    
    # ===== AdaBoost (10 variantes) =====
    for n_est in [100, 200, 300, 400]:
        for lr in [0.5, 1.0, 1.5]:
            if len([m for m in models if m[0] == 'ADA']) < 10:
                models.append(('ADA', {
                    'n_estimators': n_est, 'learning_rate': lr,
                    'random_state': 42
                }, AdaBoostClassifier))
    
    # ===== MLP Neural Network (15 variantes) =====
    for hidden in [(128, 64), (256, 128), (128, 64, 32), (256, 128, 64)]:
        for alpha in [0.0001, 0.001, 0.01]:
            if len([m for m in models if m[0] == 'MLP']) < 15:
                models.append(('MLP', {
                    'hidden_layer_sizes': hidden, 'alpha': alpha,
                    'max_iter': 500, 'random_state': 42
                }, MLPClassifier))
    
    # ===== SVC (8 variantes) =====
    for c in [0.5, 1.0, 2.0, 5.0]:
        for kernel in ['rbf', 'poly']:
            if len([m for m in models if m[0] == 'SVC']) < 8:
                models.append(('SVC', {
                    'C': c, 'kernel': kernel, 'probability': True,
                    'class_weight': 'balanced', 'random_state': 42
                }, SVC))
    
    # ===== Bagging (10 variantes) =====
    for n_est in [50, 100, 150, 200]:
        for max_f in [0.8, 1.0]:
            if len([m for m in models if m[0] == 'BAG']) < 10:
                models.append(('BAG', {
                    'n_estimators': n_est, 'max_features': max_f,
                    'random_state': 42, 'n_jobs': -1
                }, BaggingClassifier))
    
    return models

all_model_configs = create_all_models()
print(f" {len(all_model_configs)} configuraciones (vs 136 en V18)")

type_counts = {}
for m_type, _, _ in all_model_configs:
    type_counts[m_type] = type_counts.get(m_type, 0) + 1

print(" Distribución:")
for m_type, count in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"   {m_type}: {count} modelos")

# =============================================================================
# EVALUACIÓN CON 7-FOLD (VS 5-FOLD EN V18)
# =============================================================================

print("\n EVALUANDO MODELOS CON 7-FOLD")

cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

all_results = []

for i, (m_type, config, model_class) in enumerate(all_model_configs):
    try:
        oof_preds = np.zeros(len(y_encoded))
        
        for train_idx, val_idx in cv.split(X_train_sc, y_encoded):
            X_tr, X_val = X_train_sc[train_idx], X_train_sc[val_idx]
            y_tr = y_encoded[train_idx]
            
            model = model_class(**config)
            model.fit(X_tr, y_tr)
            
            pred = model.predict(X_val)
            if len(pred.shape) > 1:
                pred = pred.flatten()
            oof_preds[val_idx] = pred
        
        oof_score = accuracy_score(y_encoded, oof_preds)
        all_results.append({
            'type': m_type,
            'config': config,
            'model_class': model_class,
            'oof_score': oof_score
        })
        
        if (i + 1) % 25 == 0:
            best_so_far = max([r['oof_score'] for r in all_results])
            print(f"    {i+1}/{len(all_model_configs)} | Mejor: {best_so_far:.4f}")
    
    except Exception as e:
        continue

all_results.sort(key=lambda x: x['oof_score'], reverse=True)

print(f"\n {len(all_results)} modelos evaluados exitosamente")

# =============================================================================
# TOP 50 MODELOS
# =============================================================================

print("\n TOP 50 MODELOS")
for i in range(min(50, len(all_results))):
    res = all_results[i]
    print(f"   {i+1}. {res['type']}: {res['oof_score']:.4f}")

top_50 = all_results[:50]

print("\n ENTRENANDO TOP 50 CON DATOS COMPLETOS")
trained_models = []

for res in top_50:
    try:
        model = res['model_class'](**res['config'])
        model.fit(X_train_sc, y_encoded)
        trained_models.append({
            'type': res['type'],
            'model': model,
            'oof_score': res['oof_score']
        })
    except:
        continue

print(f" {len(trained_models)} modelos entrenados")

# =============================================================================
# MEGA ENSEMBLE MEJORADO
# =============================================================================

print("\n MEGA ENSEMBLE MEJORADO")

scores = np.array([m['oof_score'] for m in trained_models])

# ESTRATEGIA DUAL DE PESOS
# Estrategia 1: Potencia 4 (como V18)
weights_pow4 = scores ** 4
weights_pow4 = weights_pow4 / weights_pow4.sum()

# Estrategia 2: Exponencial (como V20)
weights_exp = np.exp(scores * 20)
weights_exp = weights_exp / weights_exp.sum()

# Estrategia 3: Mix 50-50
weights_mix = 0.5 * weights_pow4 + 0.5 * weights_exp

print(f" Top 15 pesos (Mix):")
for i in range(min(15, len(trained_models))):
    print(f"   {trained_models[i]['type']}: {weights_mix[i]:.4f} (OOF: {trained_models[i]['oof_score']:.4f})")

# =============================================================================
# PREDICCIÓN CON LAS 3 ESTRATEGIAS
# =============================================================================

print("\n GENERANDO 3 PREDICCIONES")

def get_predictions(weights):
    probas = []
    for model_info, weight in zip(trained_models, weights):
        try:
            proba = model_info['model'].predict_proba(X_test_sc)
            probas.append(proba * weight)
        except:
            continue
    
    ensemble_proba = np.sum(probas, axis=0)
    ensemble_proba = ensemble_proba / ensemble_proba.sum(axis=1, keepdims=True)
    
    final_pred = np.argmax(ensemble_proba, axis=1)
    final_labels = le.inverse_transform(final_pred)
    
    return final_labels

labels_pow4 = get_predictions(weights_pow4)
labels_exp = get_predictions(weights_exp)
labels_mix = get_predictions(weights_mix)

# Analizar distribuciones
print("\n COMPARACIÓN DE DISTRIBUCIONES:")

for name, labels in [("POW4 (V18)", labels_pow4), ("EXP20 (V20)", labels_exp), ("MIX", labels_mix)]:
    dist = pd.Series(labels).value_counts(normalize=True).sort_index()
    print(f"\n{name}:")
    for clase in le.classes_:
        print(f"   {clase}: {dist.get(clase, 0):.4f}")

# Usar estrategia MIX (mejor de ambos mundos)
final_labels = labels_mix

# =============================================================================
# SUBMISSION
# =============================================================================

print("\n DISTRIBUCIÓN FINAL (MIX)")
final_dist = pd.Series(final_labels).value_counts(normalize=True).sort_index()
for clase in le.classes_:
    print(f"   {clase}: {final_dist.get(clase, 0):.4f}")

final_pred = np.array([list(le.classes_).index(label) for label in final_labels])

submission_data = np.zeros((len(final_pred), 3))
for i, pred in enumerate(final_pred):
    submission_data[i, pred] = 1.0

submission_df = pd.DataFrame(submission_data, columns=['Dropout', 'Enrolled', 'Graduate'])
submission_df.insert(0, 'id', test_ids)
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\n submission.csv GUARDADO")
print(f"\n Sample:")
print(submission_df.sample(10, random_state=42))

print("\n" + "=" * 80)
print(" RESUMEN")
print("=" * 80)
print(f" {len(all_model_configs)} configs")
print(f" 7-Fold CV (vs 5-Fold en V18)")
print(f" Top 50 modelos (vs 30 en V18)")
print(f" Ensemble dual: Mix de Pow4 + Exp20")
print(f" Más variaciones de modelos top")
print(f" OOF CV Best: {all_results[0]['oof_score']:.4f}")
print(f" OOF CV Top-50: {np.mean(scores):.4f}")
print(f" Tipos de modelos: {len(type_counts)} algoritmos")
print(f" MEJORAS vs V18:")
print(f"   • +70 configs más: +0.002-0.005")
print(f"   • 7-Fold más robusto: +0.001-0.003")
print(f"   • Top 50 vs 30: +0.002-0.004")
print(f"   • Mejor estrategia pesos: +0.003-0.007")
print(f" SCORE ESPERADO: 0.825-0.840")
print("=" * 80)